# 08. 데이터 증강 실험

`train_eda_fixed.csv`만 사용한다. 데이터 증강은 train split에만 적용하고, 모델 선택과 confusion matrix는 validation 기준으로만 확인한다.

In [ ]:
from pathlib import Path
import os
import random
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

# 노트북 출력에서 반복 경고는 숨기고, 그래프 스타일은 공통으로 맞춘다.
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')


def setup_korean_font():
    # Windows, macOS, Linux/Colab 순서로 사용 가능한 한글 폰트를 찾는다.
    candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans KR']
    available = {font.name for font in fm.fontManager.ttflist}
    for font_name in candidates:
        if font_name in available:
            plt.rcParams['font.family'] = font_name
            plt.rcParams['axes.unicode_minus'] = False
            print(f'한글 폰트 설정: {font_name}')
            return font_name

    # Colab/Linux에서 Nanum 폰트가 없으면 설치를 시도한다.
    if Path('/content').exists():
        subprocess.run(['apt-get', '-qq', 'update'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
        plt.rcParams['font.family'] = 'NanumGothic'
        plt.rcParams['axes.unicode_minus'] = False
        print('한글 폰트 설정: NanumGothic')
        return 'NanumGothic'

    # 폰트가 없을 때도 마이너스 기호 깨짐은 방지한다.
    plt.rcParams['axes.unicode_minus'] = False
    print('사용 가능한 한글 폰트를 찾지 못했다. 필요하면 NanumGothic 또는 Malgun Gothic을 설치한다.')
    return None


setup_korean_font()

# 모든 실험에서 같은 난수 시드를 사용해 결과 재현성을 맞춘다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
# imblearn Pipeline을 사용해야 sampler가 train fold에만 적용된다.
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import ADASYN, RandomOverSampler, SMOTE

## 데이터 로드 및 시간순 검증 분리

In [ ]:
# 노트북 폴더와 프로젝트 루트 양쪽 실행을 모두 지원한다.
DATA_DIR = Path('../data')
if not (DATA_DIR / 'train_eda_fixed.csv').exists():
    DATA_DIR = Path('data')

TRAIN_PATH = DATA_DIR / 'train_eda_fixed.csv'
DATE_COL = 'baseline_create_date'
TARGET_COL = 'target'
VALID_SIZE = 0.2

# 모델 선택 단계에서는 test 데이터를 읽지 않고 train 데이터만 사용한다.
raw = pd.read_csv(TRAIN_PATH)
raw[DATE_COL] = pd.to_datetime(raw[DATE_COL])
raw = raw.sort_values(DATE_COL).reset_index(drop=True)

# 시계열성을 고려해 과거 80%를 학습, 이후 20%를 검증 데이터로 사용한다.
split_idx = int(len(raw) * (1 - VALID_SIZE))
train_inner_raw = raw.iloc[:split_idx].copy()
valid_raw = raw.iloc[split_idx:].copy()

print(f'전체 데이터: {raw.shape}')
print(f'train_inner: {train_inner_raw.shape} | {train_inner_raw[DATE_COL].min().date()} ~ {train_inner_raw[DATE_COL].max().date()}')
print(f'validation : {valid_raw.shape} | {valid_raw[DATE_COL].min().date()} ~ {valid_raw[DATE_COL].max().date()}')
display(pd.DataFrame({
    'train_inner': train_inner_raw[TARGET_COL].value_counts(normalize=True).sort_index(),
    'validation': valid_raw[TARGET_COL].value_counts(normalize=True).sort_index(),
}))

## 공통 피처 구성

In [ ]:
# target을 직접 설명하거나 운영 시점에 알 수 없는 컬럼은 모델 입력에서 제외한다.
DROP_COLS = [
    'cust_number', 'name_customer', 'clear_date', 'buisness_year', 'due_in_date',
    'posting_id', 'baseline_create_date', 'target_old', 'year_month', 'year_quarter',
    'business_days_late',
]


def split_xy(df):
    # 같은 컬럼 규칙으로 feature와 target을 분리한다.
    drop_cols = [col for col in DROP_COLS if col in df.columns]
    model_df = df.drop(columns=drop_cols).copy()
    y = model_df[TARGET_COL].astype(int)
    X = model_df.drop(columns=TARGET_COL)
    return X, y


X_train, y_train = split_xy(train_inner_raw)
X_valid, y_valid = split_xy(valid_raw)

# 전처리기는 train split에서 확인한 컬럼 목록을 기준으로 구성한다.
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

print(f'사용 피처 수: {X_train.shape[1]}')
print(f'수치형: {len(num_cols)} | 범주형: {len(cat_cols)}')
print('제거 컬럼:', [col for col in DROP_COLS if col in raw.columns])

In [ ]:
def make_preprocessor(scale_numeric=False):
    # 수치형은 median imputation, 범주형은 최빈값 imputation 후 one-hot encoding을 적용한다.
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        # Logistic Regression처럼 scale에 민감한 모델에만 StandardScaler를 적용한다.
        numeric_steps.append(('scaler', StandardScaler()))

    categorical_steps = [
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]

    return ColumnTransformer(
        transformers=[
            ('num', Pipeline(numeric_steps), num_cols),
            ('cat', Pipeline(categorical_steps), cat_cols),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )


def make_model_pipeline(model, scale_numeric=False):
    # 전처리와 모델을 하나의 Pipeline으로 묶어 검증 과정의 전처리 누수를 막는다.
    return Pipeline([
        ('preprocess', make_preprocessor(scale_numeric=scale_numeric)),
        ('model', model),
    ])


def evaluate_predictions(name, y_true, pred, pred_proba=None):
    # 모델 선택에 필요한 지표와 confusion matrix는 모두 validation 기준으로 출력한다.
    print('=' * 70)
    print(f'[{name}] 검증 성능')
    print(f'Accuracy : {accuracy_score(y_true, pred):.4f}')
    print(f'Macro F1 : {f1_score(y_true, pred, average="macro"):.4f}')
    if pred_proba is not None:
        print(f'ROC AUC  : {roc_auc_score(y_true, pred_proba):.4f}')
    print()
    print('분류 리포트')
    print(classification_report(y_true, pred, labels=[0, 1]))
    print('Confusion matrix')  # 행: 실제값, 열: 예측값, labels=[0, 1] 순서
    print(confusion_matrix(y_true, pred, labels=[0, 1]))


def evaluate_model(name, pipeline):
    # train_inner로 학습하고 validation으로만 성능을 확인한다.
    fitted = clone(pipeline)
    fitted.fit(X_train, y_train)
    pred = fitted.predict(X_valid)
    pred_proba = fitted.predict_proba(X_valid)[:, 1]
    evaluate_predictions(name, y_valid, pred, pred_proba)
    return {
        'model': name,
        'accuracy': accuracy_score(y_valid, pred),
        'macro_f1': f1_score(y_valid, pred, average='macro'),
        'roc_auc': roc_auc_score(y_valid, pred_proba),
        'fitted_model': fitted,
        'valid_proba': pred_proba,
    }


def find_best_threshold(y_true, pred_proba):
    # threshold도 test가 아니라 validation 예측 확률로만 선택한다.
    rows = []
    for threshold in np.arange(0.10, 0.91, 0.01):
        pred = (pred_proba >= threshold).astype(int)
        rows.append({'threshold': threshold, 'macro_f1': f1_score(y_true, pred, average='macro')})
    return pd.DataFrame(rows).sort_values('macro_f1', ascending=False).reset_index(drop=True)

## 데이터 증강 Pipeline

In [ ]:
def make_sampling_pipeline(model, sampler=None, scale_numeric=False):
    # sampler는 전처리 이후, 모델 학습 이전에만 적용된다.
    steps = [('preprocess', make_preprocessor(scale_numeric=scale_numeric))]
    if sampler is not None:
        steps.append(('sampler', sampler))
    steps.append(('model', model))
    return ImbPipeline(steps)


def evaluate_with_sampler(model_name, model_config, sampler_name, sampler):
    # 증강이 있는 경우에도 validation 데이터에는 sampler를 적용하지 않는다.
    pipeline = make_sampling_pipeline(clone(model_config['model']), clone(sampler) if sampler is not None else None, model_config['scale_numeric'])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_valid)
    proba = pipeline.predict_proba(X_valid)[:, 1]
    return {'model': model_name, 'augmentation': sampler_name, 'accuracy': accuracy_score(y_valid, pred), 'macro_f1': f1_score(y_valid, pred, average='macro'), 'roc_auc': roc_auc_score(y_valid, proba), 'fitted_model': pipeline, 'valid_proba': proba, 'valid_pred': pred}

In [ ]:
# 증강 방식 비교에서도 모델 후보와 전처리 규칙은 06/07과 동일하게 유지한다.
models = {
    'Logistic Regression': {'model': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED), 'scale_numeric': True},
    'Random Forest': {'model': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1), 'scale_numeric': False},
    'XGBoost': {'model': xgb.XGBClassifier(n_estimators=300, random_state=SEED, seed=SEED, eval_metric='logloss', n_jobs=-1), 'scale_numeric': False},
    'LightGBM': {'model': lgb.LGBMClassifier(n_estimators=300, random_state=SEED, seed=SEED, verbose=-1, n_jobs=-1), 'scale_numeric': False},
}

# None은 증강을 쓰지 않는 기준선이다. 나머지는 train split에만 적용된다.
samplers = {
    'None': None,
    'RandomOverSampler': RandomOverSampler(random_state=SEED),
    'SMOTE': SMOTE(random_state=SEED, k_neighbors=5),
    'ADASYN': ADASYN(random_state=SEED, n_neighbors=5),
}

## 검증 성능 비교

In [ ]:
# 모든 조합을 validation 성능 기준으로 비교한다.
results = []
for model_name, model_config in models.items():
    for sampler_name, sampler in samplers.items():
        print(f'[{model_name} + {sampler_name}] validation 평가')
        row = evaluate_with_sampler(model_name, model_config, sampler_name, sampler)
        results.append(row)
        print(f"  macro F1={row['macro_f1']:.4f} | ROC AUC={row['roc_auc']:.4f}")

summary_df = pd.DataFrame([{k: v for k, v in row.items() if k not in ['fitted_model', 'valid_proba', 'valid_pred']} for row in results]).sort_values('macro_f1', ascending=False).reset_index(drop=True)
display(summary_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=summary_df, x='macro_f1', y='model', hue='augmentation', ax=axes[0])
axes[0].set_title('증강 방식별 검증 Macro F1')
axes[0].set_xlim(0, 1)
sns.barplot(data=summary_df, x='roc_auc', y='model', hue='augmentation', ax=axes[1])
axes[1].set_title('증강 방식별 검증 ROC AUC')
axes[1].set_xlim(0, 1)
plt.tight_layout()

## 최적 증강 방식 및 Threshold 확인

In [ ]:
# 가장 높은 validation Macro F1 조합을 선택하고 confusion matrix를 확인한다.
best_row = summary_df.iloc[0]
best_result = next(row for row in results if row['model'] == best_row['model'] and row['augmentation'] == best_row['augmentation'])
evaluate_predictions(f"{best_result['model']} + {best_result['augmentation']}", y_valid, best_result['valid_pred'], best_result['valid_proba'])

threshold_df = find_best_threshold(y_valid, best_result['valid_proba'])
display(threshold_df.head(10))

best_threshold = threshold_df.loc[0, 'threshold']
threshold_pred = (best_result['valid_proba'] >= best_threshold).astype(int)
evaluate_predictions(f"{best_result['model']} + {best_result['augmentation']} threshold={best_threshold:.2f}", y_valid, threshold_pred, best_result['valid_proba'])